In [12]:
from pathlib import Path
import base64

import imageio.v2 as imageio
import numpy as np
from IPython.display import HTML, display
from lerobot.datasets import LeRobotDataset

# Configuration
ROOT = Path("/workspace/lerobot/outputs/datasets/libero_success_rollouts")
REPO_ID = "siwooyong/libero-smolvla-success-rollouts"
EPISODE_INDEX = 0
OUTPUT = Path("/workspace/lerobot/outputs/episode_000.mp4")

# Load dataset
dataset = LeRobotDataset(repo_id=REPO_ID, root=ROOT)

episodes = dataset.meta.episodes
start = int(episodes["dataset_from_index"][EPISODE_INDEX])
end = int(episodes["dataset_to_index"][EPISODE_INDEX])

keys = [
    "observation.images.image",
    "observation.images.image2",
]

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# Write H.264 video
with imageio.get_writer(
    str(OUTPUT),
    format="FFMPEG",
    mode="I",
    fps=float(dataset.meta.fps),
    codec="libx264",
    pixelformat="yuv420p",
    quality=8,
) as writer:
    for index in range(start, end):
        sample = dataset[index]
        frames = []

        for key in keys:
            image = sample[key]

            if image.ndim == 4:
                image = image[0]

            image = image.detach().cpu().float().clamp(0, 1)
            image = image.permute(1, 2, 0).numpy()
            image = (image * 255).astype(np.uint8)
            frames.append(image)

        frame = np.concatenate(frames, axis=1)
        writer.append_data(frame)

print(f"Saved: {OUTPUT}")
print(f"Frames: {end - start}")
print(f"FPS: {dataset.meta.fps}")

# Embed the video directly in the notebook
video_b64 = base64.b64encode(OUTPUT.read_bytes()).decode()

display(HTML(f"""
<video width="960" controls autoplay loop>
    <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
"""))

Saved: /workspace/lerobot/outputs/episode_000.mp4
Frames: 429
FPS: 10
